#  Scam-Only Linguistic Pattern Analysis
### Deep Dive Into What Makes Scams Tick


## Step 1: Install Libraries

In [ ]:
!pip install -q transformers peft torch pandas numpy scikit-learn tqdm
!pip install -q matplotlib seaborn wordcloud plotly
!pip install -q sentence-transformers umap-learn
!pip install -q nltk spacy textblob
!python -m spacy download en_core_web_sm -q

import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('vader_lexicon', quiet=True)
nltk.download('punkt_tab')


print(' All libraries installed!')

## Step 2: Imports

In [ ]:
# Core
import pandas as pd
import numpy as np
import os, glob, re, warnings
from collections import Counter, defaultdict
warnings.filterwarnings('ignore')
from tqdm import tqdm
# NLP
import nltk
import spacy
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.sentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer

# Model
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
import seaborn as sns
from wordcloud import WordCloud
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# UMAP
try:
    import umap
    UMAP_AVAILABLE = True
except:
    UMAP_AVAILABLE = False
    print('UMAP not available, using PCA')

# Styling - Dark cyberpunk aesthetic
plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0a0e27',
    'axes.facecolor': '#16213e',
    'axes.edgecolor': '#e94560',
    'axes.labelcolor': '#f1f1f1',
    'text.color': '#f1f1f1',
    'xtick.color': '#a8dadc',
    'ytick.color': '#a8dadc',
    'grid.color': '#1d3557',
    'grid.alpha': 0.3,
    'font.family': 'monospace',
    'font.size': 10
})

# Color palette
PRIMARY   = '#e94560'  # Hot pink/red
SECONDARY = '#0f3460'  # Deep blue
ACCENT1   = '#f9c74f'  # Yellow
ACCENT2   = '#90be6d'  # Green
ACCENT3   = '#4cc9f0'  # Cyan
GRADIENT  = ['#e94560', '#f77f00', '#f9c74f', '#90be6d', '#4cc9f0', '#a78bfa']

nlp = spacy.load('en_core_web_sm')
STOP_WORDS = set(stopwords.words('english'))
sia = SentimentIntensityAnalyzer()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\n Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('\nAll imports complete!')

## Step 3: Mount Drive & Configure

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print(' Drive mounted!')

In [ ]:


CLASSIFIED_FOLDER = '/content/drive/MyDrive/scam_detection/distil_ScamContent_v5'

# Your fine-tuned model
MODEL_FOLDER = '/content/drive/MyDrive/scam_detection/Qwen_lora_sms_scam/final_model'
BASE_MODEL   = "Qwen/Qwen3-0.6B"

LABEL_COLUMN = 'qwen_classification'

# Output directory
OUTPUT_DIR = '/content/drive/MyDrive/scam_detection/scam_only_analysis_v5'
os.makedirs(OUTPUT_DIR, exist_ok=True)

MIN_SCAM_DOCS = 10  # Minimum scam docs needed for analysis

print(' Configuration complete!')
print(f'   Input  : {CLASSIFIED_FOLDER}')
print(f'   Output : {OUTPUT_DIR}')
print(f'   Label  : {LABEL_COLUMN}')

## Step 4: Load Scam Data

In [ ]:
print(f'Loading CSVs from: {CLASSIFIED_FOLDER}\n')

csv_files = sorted(glob.glob(os.path.join(CLASSIFIED_FOLDER, '*.csv')))
print(f'Found {len(csv_files)} CSV files')

dfs = []
for fp in csv_files:
    try:
        df = pd.read_csv(fp)
        df['_source'] = os.path.basename(fp)
        dfs.append(df)
    except Exception as e:
        print(f'Skipped {os.path.basename(fp)}: {e}')

df_all = pd.concat(dfs, ignore_index=True)
df_all[LABEL_COLUMN] = df_all[LABEL_COLUMN].str.lower().str.strip()
df_all = df_all.dropna(subset=['text'])
df_all['text'] = df_all['text'].astype(str)

# Extract ONLY scam rows
df_scam = df_all[df_all[LABEL_COLUMN] == 'scam'].reset_index(drop=True)

print(f'\n{"="*70}')
print(f'SCAM CORPUS LOADED')
print(f'{"="*70}')
print(f'Total documents : {len(df_all):,}')
print(f'Scam documents  : {len(df_scam):,} ({len(df_scam)/len(df_all)*100:.1f}%)')
print(f'Source files    : {df_scam["_source"].nunique()}')
print(f'{"="*70}\n')

if len(df_scam) < MIN_SCAM_DOCS:
    raise ValueError(f' Only {len(df_scam)} scam docs found. Need at least {MIN_SCAM_DOCS}.')

scam_texts = df_scam['text'].tolist()
print(f'Ready to analyze {len(scam_texts):,} scam documents!')

## Step 5: Scam Text Statistics & Distribution

In [ ]:
# Compute text features
df_scam['word_count']    = df_scam['text'].apply(lambda t: len(t.split()))
df_scam['char_count']    = df_scam['text'].apply(len)
df_scam['sent_count']    = df_scam['text'].apply(lambda t: len(sent_tokenize(t)))
df_scam['exclamations']  = df_scam['text'].apply(lambda t: t.count('!'))
df_scam['questions']     = df_scam['text'].apply(lambda t: t.count('?'))
df_scam['caps_ratio']    = df_scam['text'].apply(lambda t: sum(1 for c in t if c.isupper()) / max(len(t), 1))
df_scam['avg_word_len']  = df_scam['text'].apply(lambda t: np.mean([len(w) for w in t.split()]) if len(t.split()) > 0 else 0)
df_scam['url_count']     = df_scam['text'].apply(lambda t: len(re.findall(r'http[s]?://\S+', t)))
df_scam['dollar_signs']  = df_scam['text'].apply(lambda t: t.count('$'))

stats_summary = df_scam[[
    'word_count','char_count','sent_count','exclamations',
    'questions','caps_ratio','avg_word_len','url_count','dollar_signs'
]].describe()

print('SCAM TEXT STATISTICS')
print('='*70)
print(stats_summary.round(2).to_string())
print('='*70)

In [ ]:
# Visualize distributions
fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor('#0a0e27')
gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.3)

features = [
    ('word_count', 'Word Count', 50),
    ('sent_count', 'Sentence Count', 30),
    ('exclamations', 'Exclamation Marks', 20),
    ('questions', 'Question Marks', 15),
    ('caps_ratio', 'CAPS Ratio', 30),
    ('avg_word_len', 'Avg Word Length', 25),
    ('url_count', 'URL Count', 15),
    ('dollar_signs', 'Dollar Signs ($)', 15),
    ('char_count', 'Character Count', 40)
]

for i, (col, title, bins) in enumerate(features):
    ax = fig.add_subplot(gs[i // 3, i % 3])
    ax.set_facecolor('#16213e')

    vals = df_scam[col].values
    ax.hist(vals, bins=bins, color=GRADIENT[i % len(GRADIENT)],
            alpha=0.8, edgecolor='none')

    # Add median line
    median = np.median(vals)
    ax.axvline(median, color='white', linestyle='--', linewidth=1.5, alpha=0.7)
    ax.text(median, ax.get_ylim()[1] * 0.9, f'median: {median:.1f}',
            color='white', fontsize=8, ha='center',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#0a0e27', alpha=0.7))

    ax.set_title(title, fontweight='bold', color=GRADIENT[i % len(GRADIENT)], pad=8)
    ax.set_xlabel('')
    ax.set_ylabel('Count', fontsize=9)
    ax.grid(alpha=0.2)

fig.suptitle('SCAM TEXT FEATURE DISTRIBUTIONS',
             fontsize=16, fontweight='bold', color=PRIMARY, y=0.995)
plt.savefig(os.path.join(OUTPUT_DIR, '01_text_distributions.png'),
            dpi=150, bbox_inches='tight', facecolor='#0a0e27')
plt.show()
print('Text statistics visualized!')

## Step 6: Keyword & N-gram Analysis

In [ ]:
def get_top_ngrams(texts, n=1, top_k=40, extra_stop=None):
    """Extract top n-grams using TF-IDF."""
    stop = list(STOP_WORDS)
    if extra_stop:
        stop += extra_stop

    vectorizer = TfidfVectorizer(
        ngram_range=(n, n),
        stop_words=stop,
        max_features=5000,
        min_df=2
    )
    X = vectorizer.fit_transform(texts)
    scores = np.asarray(X.mean(axis=0)).flatten()
    vocab = vectorizer.get_feature_names_out()
    top_idx = scores.argsort()[::-1][:top_k]
    return [(vocab[i], scores[i]) for i in top_idx]

# Common noise words
noise = ['www','com','http','https','html','page','website','click','link','will','one','also','get','use','said','like']

print('Extracting top keywords...\n')
top_uni   = get_top_ngrams(scam_texts, n=1, top_k=40, extra_stop=noise)
top_bi    = get_top_ngrams(scam_texts, n=2, top_k=30, extra_stop=noise)
top_tri   = get_top_ngrams(scam_texts, n=3, top_k=20, extra_stop=noise)

print('TOP 25 UNIGRAMS (TF-IDF):')
print('-'*60)
for w, s in top_uni[:25]:
    print(f'  {w:<30} {s:.4f}')

print('\nTOP 20 BIGRAMS:')
print('-'*60)
for w, s in top_bi[:20]:
    print(f'  {w:<40} {s:.4f}')

print('\nTOP 15 TRIGRAMS:')
print('-'*60)
for w, s in top_tri[:15]:
    print(f'  {w:<50} {s:.4f}')

In [ ]:
# N-gram visualization
fig, axes = plt.subplots(1, 3, figsize=(22, 8))
fig.patch.set_facecolor('#0a0e27')
fig.suptitle('SCAM KEYWORD ANALYSIS — TF-IDF Ranked',
             fontsize=15, fontweight='bold', color=PRIMARY)

datasets = [
    (top_uni[:25], 'Top 25 Unigrams', GRADIENT[0]),
    (top_bi[:20],  'Top 20 Bigrams',  GRADIENT[2]),
    (top_tri[:15], 'Top 15 Trigrams', GRADIENT[4]),
]

for ax, (data, title, color) in zip(axes, datasets):
    ax.set_facecolor('#16213e')
    words, scores = zip(*data)
    y_pos = range(len(words))

    # Gradient bars
    colors_grad = plt.cm.plasma(np.linspace(0.3, 0.9, len(words)))
    bars = ax.barh(y_pos, scores, color=colors_grad, alpha=0.9, edgecolor='none')

    ax.set_yticks(y_pos)
    ax.set_yticklabels(words, fontsize=8.5)
    ax.invert_yaxis()
    ax.set_title(title, color=color, fontweight='bold', pad=12, fontsize=13)
    ax.set_xlabel('TF-IDF Score', fontsize=10)
    ax.grid(axis='x', alpha=0.2)

    # Value labels
    for bar, score in zip(bars, scores):
        ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                f'{score:.3f}', va='center', fontsize=7, color='#a8dadc')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '02_ngram_keywords.png'),
            dpi=150, bbox_inches='tight', facecolor='#0a0e27')
plt.show()
print('N-gram analysis saved!')

In [ ]:
# Word cloud
all_text = ' '.join(scam_texts)

wc = WordCloud(
    width=1600, height=800,
    background_color='#0a0e27',
    colormap='plasma',
    max_words=150,
    stopwords=STOP_WORDS.union(set(noise)),
    prefer_horizontal=0.75,
    collocations=True,
    relative_scaling=0.5
).generate(all_text)

fig, ax = plt.subplots(figsize=(20, 10))
fig.patch.set_facecolor('#0a0e27')
ax.set_facecolor('#0a0e27')
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title(f'SCAM CORPUS WORD CLOUD\n{len(scam_texts):,} documents',
             fontsize=20, fontweight='bold', color=PRIMARY, pad=20)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '03_wordcloud.png'),
            dpi=150, bbox_inches='tight', facecolor='#0a0e27')
plt.show()
print('Word cloud generated!')

## Step 7: Discourse Strategy Detection

In [ ]:
# Define scam discourse strategies
STRATEGIES = {
    'Urgency & Time Pressure': [
        r'\burgent\b', r'\bimmediately\b', r'\bnow\b', r'\btoday\b',
        r'\blimited time\b', r'\bexpires\b', r'\bdeadline\b', r'\bhurry\b',
        r'\blast chance\b', r'\bact now\b', r'\bquickly\b', r'\binstant\b',
        r'\bfinal notice\b', r'\bonly.*hours?\b', r'\bending soon\b',
        r'\bdon.t wait\b', r'\bwhile supplies last\b'
    ],
    'Reward & Financial Gain': [
        r'\bwon\b', r'\bwinner\b', r'\bprize\b', r'\breward\b',
        r'\bcongratulations\b', r'\bclaim\b', r'\bgift\b', r'\bcash\b',
        r'\$\d+', r'\bfree\b', r'\bbonus\b', r'\bjackpot\b', r'\bgiveaway\b',
        r'\blottery\b', r'\bselected\b', r'\bprofit\b', r'\bearnings\b',
        r'\bincome\b', r'\bguaranteed\b'
    ],
    'Authority Impersonation': [
        r'\bgovernment\b', r'\birs\b', r'\bfbi\b', r'\bbank\b',
        r'\bofficial\b', r'\badministration\b', r'\bauthority\b',
        r'\bceo\b', r'\bdirector\b', r'\bverify your account\b',
        r'\byour account\b', r'\bsupport team\b', r'\bcustomer service\b',
        r'\bsecurity\b', r'\bdepartment\b', r'\bagency\b'
    ],
    'Threat & Consequence': [
        r'\bsuspended\b', r'\bterminated\b', r'\blegal action\b',
        r'\bwarning\b', r'\bpenalty\b', r'\bfine\b', r'\barrest\b',
        r'\bfailed to\b', r'\bblocked\b', r'\bfrozen\b', r'\bexpired\b',
        r'\binvestigation\b', r'\blawsuit\b', r'\bprosecuted\b',
        r'\bconsequences\b', r'\bviolation\b'
    ],
    'Personal Info Request': [
        r'\bsocial security\b', r'\bssn\b', r'\bpassword\b',
        r'\bcredit card\b', r'\bbank account\b', r'\bpersonal (details|information)\b',
        r'\bdate of birth\b', r'\bverify your\b', r'\bconfirm your\b',
        r'\bprovide your\b', r'\bsend your\b', r'\bpin\b', r'\bcvv\b',
        r'\baccount number\b'
    ],
    'Investment & High Returns': [
        r'\binvest\b', r'\bprofit\b', r'\breturn\b', r'\b\d+%\b',
        r'\bcrypto\b', r'\bbitcoin\b', r'\bforex\b', r'\btrading\b',
        r'\bpassive income\b', r'\bfinancial freedom\b', r'\bwealth\b',
        r'\bguaranteed\b', r'\brisk.free\b', r'\bopportunity\b',
        r'\bmillionaire\b'
    ],
    'Social Proof & FOMO': [
        r'\bthousands\b', r'\bmillions\b', r'\beveryone\b', r'\bpeople are\b',
        r'\bdon.t miss\b', r'\blimited spots\b', r'\bexclusive\b',
        r'\bonly \d+ left\b', r'\bjoining\b', r'\balready\b',
        r'\btrending\b', r'\bviral\b'
    ],
    'Emotional Manipulation': [
        r'\bcongratulations\b', r'\blucky\b', r'\bfortunate\b',
        r'\bdream\b', r'\blife.?changing\b', r'\bamazing\b',
        r'\bincredible\b', r'\bunbelievable\b', r'\bmiracle\b',
        r'\bsecret\b', r'\bhidden\b'
    ]
}

def count_strategy_patterns(text, patterns):
    text_lower = text.lower()
    matches = []
    for p in patterns:
        if re.search(p, text_lower):
            matches.append(p)
    return len(matches), matches

# Score each scam on each strategy
print('Analyzing discourse strategies...\n')
strategy_data = {}

for strat, patterns in STRATEGIES.items():
    df_scam[f'strategy_{strat}'] = df_scam['text'].apply(
        lambda t: count_strategy_patterns(t, patterns)[0]
    )

    total_hits = df_scam[f'strategy_{strat}'].sum()
    docs_with_strategy = (df_scam[f'strategy_{strat}'] > 0).sum()
    pct_docs = (docs_with_strategy / len(df_scam)) * 100
    avg_hits = df_scam[f'strategy_{strat}'].mean()

    strategy_data[strat] = {
        'total_hits': total_hits,
        'docs_with': docs_with_strategy,
        'pct_docs': pct_docs,
        'avg_hits': avg_hits
    }

print('STRATEGY PREVALENCE IN SCAM CORPUS')
print('='*70)
print(f'{"Strategy":<35} {"% Docs":>10} {"Avg Hits":>12} {"Total":>10}')
print('-'*70)
for strat, data in sorted(strategy_data.items(), key=lambda x: x[1]['pct_docs'], reverse=True):
    print(f'{strat:<35} {data["pct_docs"]:>9.1f}% {data["avg_hits"]:>11.2f} {data["total_hits"]:>10}')
print('='*70)

In [ ]:
# Strategy prevalence chart
strategies_sorted = sorted(strategy_data.items(), key=lambda x: x[1]['pct_docs'], reverse=True)
strat_names = [s for s, _ in strategies_sorted]
pct_values = [d['pct_docs'] for _, d in strategies_sorted]

fig, ax = plt.subplots(figsize=(14, 9))
fig.patch.set_facecolor('#0a0e27')
ax.set_facecolor('#16213e')

colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(strat_names)))
bars = ax.barh(range(len(strat_names)), pct_values, color=colors, alpha=0.9, edgecolor='none')

ax.set_yticks(range(len(strat_names)))
ax.set_yticklabels(strat_names, fontsize=11)
ax.invert_yaxis()
ax.set_xlabel('% of Scam Documents Using This Strategy', fontsize=11)
ax.set_title('SCAM DISCOURSE STRATEGIES\nPrevalence Across Corpus',
             fontsize=14, fontweight='bold', color=PRIMARY, pad=15)
ax.grid(axis='x', alpha=0.2)
ax.set_xlim(0, 105)

# Value labels
for bar, val in zip(bars, pct_values):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '04_strategy_prevalence.png'),
            dpi=150, bbox_inches='tight', facecolor='#0a0e27')
plt.show()
print('Strategy analysis saved!')

## Step 8: Semantic Theme Clustering

### Determine Optimal Clusters with Silhouette Score
Evaluate $k$ values to find the best separation among semantic themes.

In [ ]:
print('Calculating silhouette scores to find optimal k...')
silhouette_scores = []
k_values = list(range(2, 12))
print('Creating sentence embeddings...\n')

embed_model = SentenceTransformer('all-mpnet-base-v2')
texts_short = [t[:512] for t in scam_texts]  # Truncate for speed

embeddings = embed_model.encode(
    texts_short,
    batch_size=32,
    show_progress_bar=True,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

print(f'\nEmbeddings: {embeddings.shape}')

# Sample embeddings for silhouette score computation to save time (O(N^2) complexity)
sample_size = min(5000, embeddings.shape[0])
np.random.seed(42)
indices = np.random.choice(embeddings.shape[0], sample_size, replace=False)
sampled_embeddings = embeddings[indices]

for k in tqdm(k_values, desc='Evaluating k'):
    kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans_temp.fit_predict(sampled_embeddings)
    score = silhouette_score(sampled_embeddings, cluster_labels)
    silhouette_scores.append(score)

N_CLUSTERS = k_values[int(np.argmax(silhouette_scores))]
print(f'\nOptimal number of clusters (highest silhouette): {N_CLUSTERS}')
print(f'Silhouette score at k={N_CLUSTERS}: {max(silhouette_scores):.4f}')
for k, s in zip(k_values, silhouette_scores):
    marker = '  <-- optimal' if k == N_CLUSTERS else ''
    print(f'  k={k:2d}:  {s:.4f}{marker}')

# Plot the silhouette scores (white background, black labels - paper ready)
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
ax.plot(k_values, silhouette_scores, marker='o', color='#1f4e79',
        linewidth=2.5, markersize=9, markerfacecolor='#1f4e79', markeredgecolor='black')
ax.axvline(N_CLUSTERS, color='#c00000', linestyle='--', linewidth=2,
           label=f'Optimal k = {N_CLUSTERS}')

# Highlight the optimal point
opt_idx = k_values.index(N_CLUSTERS)
ax.scatter([N_CLUSTERS], [silhouette_scores[opt_idx]], s=220,
           facecolor='none', edgecolor='#c00000', linewidth=2.5, zorder=5)

ax.set_title('Silhouette Score vs. Number of Clusters',
             color='black', fontweight='bold', pad=15, fontsize=18)
ax.set_xlabel('Number of Clusters (k)', color='black', fontsize=15, fontweight='bold')
ax.set_ylabel('Silhouette Score', color='black', fontsize=15, fontweight='bold')
ax.set_xticks(k_values)
ax.tick_params(axis='both', labelsize=13, colors='black')
for spine in ax.spines.values():
    spine.set_color('black')
ax.grid(alpha=0.3, color='gray')
ax.legend(fontsize=14, facecolor='white', edgecolor='black', labelcolor='black')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '04_silhouette_scores.png'),
            dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print('Silhouette plot saved!')



print(f'\nRe-clustering with silhouette-selected k={N_CLUSTERS} (was {N_CLUSTERS})...')
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
clusters = kmeans.fit_predict(embeddings)



In [ ]:


# Cluster
print(f'Clustering into {N_CLUSTERS} semantic themes...')
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
clusters = kmeans.fit_predict(embeddings)
df_scam['cluster'] = clusters

# 2D projection
if UMAP_AVAILABLE:
    reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
    coords_2d = reducer.fit_transform(embeddings)
    method = 'UMAP'
else:
    pca = PCA(n_components=2, random_state=42)
    coords_2d = pca.fit_transform(embeddings)
    method = 'PCA'

print(f'Using {method} for visualization')

# Extract theme keywords per cluster
print('\nCLUSTER THEMES:')
print('='*70)
cluster_themes = {}
for cid in range(N_CLUSTERS):
    cluster_texts = df_scam[df_scam['cluster'] == cid]['text'].tolist()
    if len(cluster_texts) < 3:
        cluster_themes[cid] = 'insufficient data'
        continue
    top_terms = get_top_ngrams(cluster_texts, n=1, top_k=8, extra_stop=noise)
    theme = ', '.join([w for w, _ in top_terms[:8]])
    cluster_themes[cid] = theme
    print(f'Cluster {cid} ({len(cluster_texts)} docs): {theme}')
print('='*70)

In [ ]:
# Plot clusters (white background, larger black labels - paper ready)
fig, ax = plt.subplots(figsize=(16, 12))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

palette = plt.cm.tab10(np.linspace(0, 1, N_CLUSTERS))

for cid in range(N_CLUSTERS):
    mask = clusters == cid
    ax.scatter(
        coords_2d[mask, 0], coords_2d[mask, 1],
        c=[palette[cid]], label=f'Theme {cid}',
        alpha=0.7, s=70, edgecolors='black', linewidth=0.4
    )

    # Centroid annotation - black text on white box for readability
    cx, cy = coords_2d[mask].mean(axis=0)
    theme_short = cluster_themes[cid].split(', ')[:3]
    theme_text = f"T{cid}\n{', '.join(theme_short)}"
    ax.annotate(
        theme_text, (cx, cy),
        fontsize=14, color='black', ha='center', fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.5', fc='white', ec=palette[cid], lw=2.5, alpha=0.95)
    )

ax.set_title(f'Scam Semantic Themes\n{method} Projection of {len(scam_texts):,} Documents',
             fontsize=20, fontweight='bold', color='black', pad=15)
ax.set_xlabel(f'{method} Dimension 1', fontsize=16, fontweight='bold', color='black')
ax.set_ylabel(f'{method} Dimension 2', fontsize=16, fontweight='bold', color='black')
ax.tick_params(axis='both', labelsize=13, colors='black')
for spine in ax.spines.values():
    spine.set_color('black')
ax.legend(loc='upper right', fontsize=14, framealpha=0.95, edgecolor='black')
ax.grid(alpha=0.2, color='gray')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '05_semantic_clusters.png'),
            dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print('Cluster visualization saved!')


In [ ]:
# Per-cluster word clouds
n_rows = (N_CLUSTERS + 2) // 3
fig, axes = plt.subplots(n_rows, 3, figsize=(22, n_rows * 6))
fig.patch.set_facecolor('#0a0e27')
fig.suptitle('PER-THEME WORD CLOUDS', fontsize=16, fontweight='bold', color=PRIMARY)

axes = axes.flatten() if N_CLUSTERS > 3 else [axes] if N_CLUSTERS == 1 else axes
cmaps = ['Reds','Oranges','YlOrBr','Greens','Blues','Purples','pink','cool','autumn']

for cid in range(N_CLUSTERS):
    ax = axes[cid]
    ax.set_facecolor('#0a0e27')

    cluster_texts = df_scam[df_scam['cluster'] == cid]['text'].tolist()
    if len(cluster_texts) == 0:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', color='white')
        ax.axis('off')
        continue

    combined = ' '.join(cluster_texts)
    wc = WordCloud(
        width=700, height=400,
        background_color='#0a0e27',
        colormap=cmaps[cid % len(cmaps)],
        max_words=80,
        stopwords=STOP_WORDS,
        collocations=False
    ).generate(combined)

    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    theme_label = cluster_themes[cid][:60] + '...' if len(cluster_themes[cid]) > 60 else cluster_themes[cid]
    ax.set_title(
        f'Theme {cid} — {len(cluster_texts)} docs\n{theme_label}',
        color=palette[cid], fontsize=11, fontweight='bold', pad=10
    )

# Hide unused subplots
for i in range(N_CLUSTERS, len(axes)):
    axes[i].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '06_cluster_wordclouds.png'),
            dpi=150, bbox_inches='tight', facecolor='#0a0e27')
plt.show()
print('Cluster word clouds saved!')

## Step 9: Scam Subtype Taxonomy
Automatically classify scam types based on patterns

In [ ]:
# Define scam subtype rules
SCAM_SUBTYPES = {
    'Prize/Lottery': [
        r'\b(won|winner|prize|lottery|jackpot|sweepstakes|drawn)\b',
        r'\b(claim|congratulations|selected|lucky)\b'
    ],
    'Investment/Crypto': [
        r'\b(bitcoin|crypto|forex|trading|invest|profit|return)\b',
        r'\b(passive income|financial freedom|\d+% return)\b'
    ],
    'Phishing/Account': [
        r'\b(verify|confirm|suspended|account|security|update)\b',
        r'\b(unusual activity|click here|sign in)\b'
    ],
    'Romance/Dating': [
        r'\b(love|relationship|meet|date|lonely|romance)\b',
        r'\b(beautiful|attractive|interested|message me)\b'
    ],
    'Job/Employment': [
        r'\b(job|work from home|earn|opportunity|hiring)\b',
        r'\b(easy money|part.time|flexible hours)\b'
    ],
    'Tech Support': [
        r'\b(virus|infected|malware|security alert|microsoft)\b',
        r'\b(computer|windows|support|technical|fix)\b'
    ],
    'Charity/Donation': [
        r'\b(charity|donate|donation|help|support|cause)\b',
        r'\b(disaster|relief|emergency|urgent help)\b'
    ],
    'Tax/Government': [
        r'\b(irs|tax|refund|government|social security)\b',
        r'\b(owe|debt|unpaid|penalty)\b'
    ]
}

def classify_scam_subtype(text):
    text_lower = text.lower()
    scores = {}
    for subtype, patterns in SCAM_SUBTYPES.items():
        score = sum(1 for p in patterns if re.search(p, text_lower))
        if score > 0:
            scores[subtype] = score

    if not scores:
        return 'Other/Unclassified'
    return max(scores.items(), key=lambda x: x[1])[0]

print('Classifying scam subtypes...\n')
df_scam['subtype'] = df_scam['text'].apply(classify_scam_subtype)

subtype_counts = df_scam['subtype'].value_counts()
print('SCAM SUBTYPE DISTRIBUTION')
print('='*50)
for subtype, count in subtype_counts.items():
    pct = (count / len(df_scam)) * 100
    print(f'{subtype:<25} {count:>6} ({pct:>5.1f}%)')
print('='*50)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

colors_pie = plt.cm.tab20(np.linspace(0, 1, len(subtype_counts)))

wedges, texts, autotexts = ax.pie(
    subtype_counts.values,
    autopct='%1.1f%%',
    colors=colors_pie,
    startangle=90,
    pctdistance=0.80,
    wedgeprops={'linewidth': 1.5, 'edgecolor': 'white'}
)

# Percentage labels: BIG and BLACK (visible on white background)
for autotext in autotexts:
    autotext.set_color('black')
    autotext.set_fontweight('bold')
    autotext.set_fontsize(15)

# Hide the default in-wedge labels to avoid overlap (names go in legend)
for text in texts:
    text.set_visible(False)

ax.set_title('Scam Subtype Taxonomy\nAutomatic Classification',
             fontsize=22, fontweight='bold', color='black', pad=20)

# Legend with BIG BLACK text on white
ax.legend(
    wedges,
    [f'{label} ({count} docs)' for label, count in subtype_counts.items()],
    title='Scam Subtypes',
    loc='center left',
    bbox_to_anchor=(1.0, 0.5),
    frameon=False,
    fontsize=15,
    title_fontsize=17,
    labelcolor='black'
)
# Make legend title black and bold
leg = ax.get_legend()
leg.get_title().set_color('black')
leg.get_title().set_fontweight('bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '07_scam_subtypes.png'),
            dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print('Scam subtype taxonomy saved!')


## Step 10: Model Attention Analysis

In [ ]:
print('Loading model for attention analysis...\n')

try:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    base_model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL, num_labels=2, output_attentions=True
    )
    model = PeftModel.from_pretrained(base_model, MODEL_FOLDER)
    model.to(device)
    model.eval()
    print('Model loaded!')
    MODEL_LOADED = True
except Exception as e:
    print(f'Could not load model: {e}')
    print('Skipping attention analysis.')
    MODEL_LOADED = False

In [ ]:
if MODEL_LOADED:
    def get_attention_scores(text, model, tokenizer, max_len=128):
        inputs = tokenizer(text, return_tensors='pt', truncation=True,
                          max_length=max_len, padding=False)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

        with torch.no_grad():
            outputs = model(**inputs, output_attentions=True)

        if outputs.attentions is None:
            return tokens, np.ones(len(tokens))

        attn_stack = torch.stack(outputs.attentions, dim=0)
        avg_attn = attn_stack.mean(dim=[0, 1, 2])  # avg across layers, batch, heads
        token_importance = avg_attn.mean(dim=0).cpu().numpy()

        return tokens, token_importance

    def aggregate_attention(texts, model, tokenizer, top_n=30):
        token_scores = Counter()
        token_counts = Counter()

        for text in tqdm(texts, desc='Attention analysis'):
            try:
                tokens, scores = get_attention_scores(text, model, tokenizer)
                for tok, score in zip(tokens, scores):
                    clean = tok.replace('Ġ', '').replace('▁', '').lower()
                    if clean in STOP_WORDS or len(clean) < 3:
                        continue
                    if tok.startswith('<') and tok.endswith('>'):
                        continue
                    token_scores[clean] += float(score)
                    token_counts[clean] += 1
            except:
                continue

        normalized = {
            tok: token_scores[tok] / token_counts[tok]
            for tok in token_scores if token_counts[tok] >= 3
        }
        return sorted(normalized.items(), key=lambda x: x[1], reverse=True)[:top_n]

    # Run on sample
    sample_size = min(150, len(scam_texts))
    sample_texts = scam_texts[:sample_size]

    top_attn = aggregate_attention(sample_texts, model, tokenizer, top_n=30)

    print('\nTOP 25 TOKENS BY MODEL ATTENTION:')
    print('='*60)
    for tok, score in top_attn[:25]:
        print(f'  {tok:<30} {score:.5f}')
    print('='*60)
else:
    print('Attention analysis skipped.')

In [ ]:
if MODEL_LOADED and top_attn:
    tokens, scores = zip(*top_attn[:30])

    fig, ax = plt.subplots(figsize=(14, 10))
    fig.patch.set_facecolor('#0a0e27')
    ax.set_facecolor('#16213e')

    colors_grad = plt.cm.plasma(np.linspace(0.3, 0.95, len(tokens)))
    bars = ax.barh(range(len(tokens)), scores, color=colors_grad, alpha=0.9)

    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens, fontsize=10)
    ax.invert_yaxis()
    ax.set_xlabel('Average Attention Weight', fontsize=11)
    ax.set_title(
        'MODEL ATTENTION ANALYSIS\nWhat Your Qwen Model Focuses On in Scam Texts',
        fontsize=14, fontweight='bold', color=PRIMARY, pad=15
    )
    ax.grid(axis='x', alpha=0.2)

    for bar, score in zip(bars, scores):
        ax.text(bar.get_width() + 0.0002, bar.get_y() + bar.get_height()/2,
                f'{score:.4f}', va='center', fontsize=8, color='#a8dadc')

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, '08_attention_weights.png'),
                dpi=150, bbox_inches='tight', facecolor='#0a0e27')
    plt.show()
    print('Attention analysis saved!')

## Step 11: Temporal & Psychological Language

In [ ]:
# Temporal expressions
TEMPORAL_PATTERNS = {
    'Immediate': [r'\bnow\b', r'\bimmediately\b', r'\binstant\b', r'\bright now\b'],
    'Today': [r'\btoday\b', r'\btonight\b', r'\bthis evening\b'],
    'Hours': [r'\d+ hours?\b', r'\bnext \d+ hours?\b', r'\bwithin hours\b'],
    'Days': [r'\d+ days?\b', r'\bexpires in \d+ days?\b', r'\bwithin days\b'],
    'Limited Window': [r'\blimited time\b', r'\bexpires\b', r'\bending soon\b', r'\bdeadline\b'],
}

# Psychological triggers
PSYCH_TRIGGERS = {
    'Scarcity': [r'\blimited\b', r'\bonly \d+\b', r'\blast\b', r'\bfew left\b', r'\balmost gone\b'],
    'Authority': [r'\bofficial\b', r'\bgovernment\b', r'\bcertified\b', r'\bapproved\b'],
    'Social Proof': [r'\bthousands\b', r'\bmillions\b', r'\bpeople\b', r'\beveryone\b'],
    'Reciprocity': [r'\bfree\b', r'\bgift\b', r'\bbonus\b', r'\bcomplimentary\b'],
    'Fear': [r'\blose\b', r'\bmiss\b', r'\bexpire\b', r'\bterminate\b', r'\bsuspend\b'],
}

def count_pattern_category(text, patterns):
    text_lower = text.lower()
    return sum(1 for p in patterns if re.search(p, text_lower))

print('Analyzing temporal and psychological patterns...\n')

# Temporal
for cat, patterns in TEMPORAL_PATTERNS.items():
    df_scam[f'temporal_{cat}'] = df_scam['text'].apply(
        lambda t: count_pattern_category(t, patterns)
    )

# Psychological
for cat, patterns in PSYCH_TRIGGERS.items():
    df_scam[f'psych_{cat}'] = df_scam['text'].apply(
        lambda t: count_pattern_category(t, patterns)
    )

# Compute prevalence
temporal_stats = {
    cat: (df_scam[f'temporal_{cat}'] > 0).sum() / len(df_scam) * 100
    for cat in TEMPORAL_PATTERNS
}

psych_stats = {
    cat: (df_scam[f'psych_{cat}'] > 0).sum() / len(df_scam) * 100
    for cat in PSYCH_TRIGGERS
}

print('TEMPORAL LANGUAGE PREVALENCE')
print('='*50)
for cat, pct in sorted(temporal_stats.items(), key=lambda x: x[1], reverse=True):
    print(f'{cat:<20} {pct:>6.1f}%')

print('\nPSYCHOLOGICAL TRIGGERS PREVALENCE')
print('='*50)
for cat, pct in sorted(psych_stats.items(), key=lambda x: x[1], reverse=True):
    print(f'{cat:<20} {pct:>6.1f}%')
print('='*50)

In [ ]:
# Combined visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor('#0a0e27')

# Temporal
ax1.set_facecolor('#16213e')
temp_cats = list(temporal_stats.keys())
temp_vals = list(temporal_stats.values())
colors1 = plt.cm.Oranges(np.linspace(0.4, 0.9, len(temp_cats)))
ax1.barh(temp_cats, temp_vals, color=colors1, alpha=0.9)
ax1.set_xlabel('% of Scam Documents', fontsize=11)
ax1.set_title('TEMPORAL PRESSURE PATTERNS', fontsize=13, fontweight='bold', color=ACCENT1, pad=12)
ax1.grid(axis='x', alpha=0.2)
ax1.invert_yaxis()

# Psychological
ax2.set_facecolor('#16213e')
psych_cats = list(psych_stats.keys())
psych_vals = list(psych_stats.values())
colors2 = plt.cm.Purples(np.linspace(0.4, 0.9, len(psych_cats)))
ax2.barh(psych_cats, psych_vals, color=colors2, alpha=0.9)
ax2.set_xlabel('% of Scam Documents', fontsize=11)
ax2.set_title('PSYCHOLOGICAL MANIPULATION TRIGGERS', fontsize=13, fontweight='bold', color='#a78bfa', pad=12)
ax2.grid(axis='x', alpha=0.2)
ax2.invert_yaxis()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '09_temporal_psychological.png'),
            dpi=150, bbox_inches='tight', facecolor='#0a0e27')
plt.show()
print('Temporal & psychological analysis saved!')

## Step 12: Imperative Verbs & Action Language

In [ ]:
print('Extracting imperative verbs...\n')

imperative_verbs = Counter()
sample_size = min(300, len(scam_texts))

for text in tqdm(scam_texts[:sample_size], desc='POS tagging'):
    doc = nlp(text[:1000])
    for sent in doc.sents:
        tokens = list(sent)
        if tokens and tokens[0].pos_ == 'VERB' and tokens[0].dep_ == 'ROOT':
            if not any(t.dep_ == 'nsubj' for t in tokens):
                imperative_verbs[tokens[0].lemma_.lower()] += 1

top_imperatives = imperative_verbs.most_common(20)

print('TOP 20 IMPERATIVE VERBS:')
print('='*50)
for verb, count in top_imperatives:
    print(f'{verb:<25} {count:>6}')
print('='*50)

In [ ]:
if top_imperatives:
    verbs, counts = zip(*top_imperatives)

    fig, ax = plt.subplots(figsize=(12, 9))
    fig.patch.set_facecolor('#0a0e27')
    ax.set_facecolor('#16213e')

    colors_imp = plt.cm.Reds(np.linspace(0.4, 0.95, len(verbs)))
    bars = ax.barh(range(len(verbs)), counts, color=colors_imp, alpha=0.9)

    ax.set_yticks(range(len(verbs)))
    ax.set_yticklabels(verbs, fontsize=10.5)
    ax.invert_yaxis()
    ax.set_xlabel('Occurrences', fontsize=11)
    ax.set_title('IMPERATIVE VERBS IN SCAM TEXTS\nDirect Action Commands',
                 fontsize=14, fontweight='bold', color=PRIMARY, pad=15)
    ax.grid(axis='x', alpha=0.2)

    for bar, cnt in zip(bars, counts):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'{cnt}', va='center', fontsize=9)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, '10_imperative_verbs.png'),
                dpi=150, bbox_inches='tight', facecolor='#0a0e27')
    plt.show()
    print('Imperative verb analysis saved!')